In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('ratings.csv')

In [3]:
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
df.drop('timestamp', axis= 1, inplace=True)

In [5]:
movieId_counts = df['movieId'].value_counts()

valid_match_ids = movieId_counts[movieId_counts >= 200].index
filtered_df = df[df['movieId'].isin(valid_match_ids)]

In [6]:
filtered_df

,userId,movieId,rating
0,1,1,4.0
3,1,47,5.0
4,1,50,5.0
7,1,110,4.0
15,1,260,5.0
...,...,...,...
99607,610,1196,5.0
99609,610,1198,5.0
99684,610,2571,5.0
99690,610,2858,3.5


In [7]:
user_item_matrix = filtered_df.pivot_table(index='userId', columns='movieId', values='rating')
user_item_matrix

movieId,1,47,50,110,150,260,296,318,356,480,527,589,593,780,1196,1198,2571,2858,2959
userId,,,,,,,,,,,,,,,,,,,
1,4.0,5.0,5.0,4.0,NaN,5.0,3.0,NaN,4.0,4.0,5.0,NaN,4.0,3.0,5.0,5.0,5.0,5.0,5.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2.0,NaN,NaN,NaN,5.0,1.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,5.0,3.0,1.0,5.0,2.0
5,4.0,NaN,4.0,4.0,3.0,NaN,5.0,3.0,NaN,NaN,5.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,2.5,3.0,4.5,3.5,NaN,4.5,5.0,3.5,4.0,2.5,5.0,3.5,4.5,2.5,4.5,3.5,5.0,4.5,5.0
607,4.0,NaN,NaN,5.0,5.0,3.0,3.0,5.0,NaN,4.0,5.0,4.0,5.0,4.0,3.0,NaN,5.0,3.0,NaN
608,2.5,4.5,4.5,4.0,2.0,3.5,5.0,4.5,3.0,3.0,4.0,3.0,4.0,3.0,4.0,NaN,5.0,5.0,5.0


In [8]:
from implicit.als import AlternatingLeastSquares
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix

In [9]:
new_matr = user_item_matrix.fillna(0)
my_array = new_matr.values
my_array

array([[4. , 5. , 5. , ..., 5. , 5. , 5. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       ...,
       [2.5, 4.5, 4.5, ..., 5. , 5. , 5. ],
       [3. , 0. , 0. , ..., 0. , 0. , 0. ],
       [5. , 5. , 4. , ..., 5. , 3.5, 5. ]])

In [10]:
M = csr_matrix(my_array)

In [11]:
M[:10, :10].todense()

matrix([[4. , 5. , 5. , 4. , 0. , 5. , 3. , 0. , 4. , 4. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 3. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 2. , 0. , 0. , 0. , 5. , 1. , 0. , 0. , 0. ],
        [4. , 0. , 4. , 4. , 3. , 0. , 5. , 3. , 0. , 0. ],
        [0. , 4. , 1. , 5. , 4. , 0. , 2. , 5. , 5. , 5. ],
        [4.5, 0. , 4.5, 0. , 4.5, 5. , 0. , 0. , 5. , 5. ],
        [0. , 4. , 5. , 3. , 4. , 0. , 4. , 5. , 3. , 4. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 3.5, 0. ]])

In [12]:
XGT = (M > 0.5).astype(int)
XGT[:10, :10].todense()

matrix([[1, 1, 1, 1, 0, 1, 1, 0, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 1, 1, 0, 0, 0],
        [1, 0, 1, 1, 1, 0, 1, 1, 0, 0],
        [0, 1, 1, 1, 1, 0, 1, 1, 1, 1],
        [1, 0, 1, 0, 1, 1, 0, 0, 1, 1],
        [0, 1, 1, 1, 1, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 1, 0]])

In [13]:
train, test = train_test_split(XGT.T, test_size=.25, random_state=42)

In [14]:
model = AlternatingLeastSquares(factors=64)
model.fit(train)

C:\Users\User\AppData\Roaming\Python\Python311\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

In [15]:
from implicit.evaluation import precision_at_k

In [16]:
precision_at_k(model, train, test)

  0%|          | 0/5 [00:00<?, ?it/s]

0.36

In [17]:
def novelty(train_data, preds):
    item_sum = (train_data > 0.5).sum(axis=0)
    item_proba = item_sum / item_sum.sum()
    result = []
    for x in preds[0]:
        for y in x:
            result.append(np.log2(1+item_proba[0, y]))
    
    return np.mean(result)

In [18]:
user_ids = list(range(test.shape[0]))

In [19]:
predictions = model.recommend(user_ids, test)

In [20]:
predictions[0]

array([[506,  57, 364, 294,  49,  39, 103,   3, 166, 376],
       [533, 548, 244, 169, 135, 180, 453, 160, 224, 346],
       [121,  81, 493, 105,  45,   0, 533, 244, 280, 477],
       [ 79, 299, 523, 357,  88, 218, 335,  73, 199, 354],
       [320, 322, 423, 293, 494, 473, 381,  94, 291, 225]], dtype=int32)

In [21]:
novelty(train, predictions)

np.float64(0.002843586751341829)

In [22]:
movie_ids = df['movieId'].unique()

def scale_movie_id(movie_id):
    scaled = np.where(movie_ids == movie_id)[0][0] + 1
    return scaled

df['movieId'] = df['movieId'].apply(scale_movie_id)
df.head()

,userId,movieId,rating
0,1,1,4.0
1,1,2,4.0
2,1,3,4.0
3,1,4,5.0
4,1,5,5.0


In [23]:
recom_matrix = df.pivot_table(index='userId', columns='movieId', values='rating')
recom_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,9715,9716,9717,9718,9719,9720,9721,9722,9723,9724
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,4.0,4.0,5.0,5.0,3.0,5.0,4.0,5.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,4.0,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,2.5,NaN,NaN,3.0,4.5,4.0,NaN,3.5,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607,4.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,2.5,2.0,NaN,4.5,4.5,3.0,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
matr = recom_matrix.fillna(0)
rec_array = matr.values
rec_array

array([[4. , 4. , 4. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       ...,
       [2.5, 2. , 0. , ..., 0. , 0. , 0. ],
       [3. , 0. , 0. , ..., 0. , 0. , 0. ],
       [5. , 0. , 5. , ..., 3. , 3.5, 3.5]])

In [25]:
M_rec = csr_matrix(rec_array)

In [26]:
M_rec[:10, :10].todense()

matrix([[4. , 4. , 4. , 5. , 5. , 3. , 5. , 4. , 5. , 5. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 2. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [4. , 0. , 0. , 0. , 4. , 0. , 0. , 4. , 0. , 0. ],
        [0. , 5. , 4. , 4. , 1. , 0. , 0. , 5. , 4. , 0. ],
        [4.5, 0. , 0. , 0. , 4.5, 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 4. , 5. , 0. , 0. , 3. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]])

In [27]:
XGT_rec = (M_rec > 0.5).astype(int)
XGT_rec[:10, :10].todense()

matrix([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 1, 0, 0, 1, 0, 0],
        [0, 1, 1, 1, 1, 0, 0, 1, 1, 0],
        [1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
user_similarity = cosine_similarity(recom_matrix.fillna(0))
user_similarity_df = pd.DataFrame(user_similarity, index=recom_matrix.index, columns=recom_matrix.index)
user_similarity_df

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,0.164191,0.028429,0.012993,0.200395,0.106435,0.102123,0.200035,0.099388,0.075898,0.088963,...,0.178084,0.116534,0.300669,0.066032,0.148141,1.000000,0.153063,0.262558,0.069622,0.201104
607,0.269389,0.012948,0.019247,0.131746,0.152866,0.162182,0.186114,0.185142,0.011844,0.010451,...,0.092525,0.199910,0.203540,0.137834,0.118780,0.153063,1.000000,0.283081,0.149190,0.139114
608,0.291097,0.046211,0.021128,0.149858,0.135535,0.178809,0.323541,0.187233,0.100435,0.077424,...,0.158355,0.197514,0.232771,0.155306,0.178142,0.262558,0.283081,1.000000,0.121993,0.322055


In [30]:
def get_similar_users(user, top_n=2):
    similar_users = user_similarity_df.loc[user].sort_values(ascending=False)
    similar_users = similar_users.drop(user, errors='ignore')
    return similar_users.head(top_n)

def recommend_items(user, top_n=2):
    similar_users = get_similar_users(user)
    similar_users_df = recom_matrix.loc[similar_users.index]
    
    mean_ratings = similar_users_df.mean(axis=0)
    
    user_rated_items = recom_matrix.loc[user]
    user_rated_items = user_rated_items[user_rated_items > 0].index
    
    recommended_items = mean_ratings.drop(user_rated_items, errors='ignore')

    return recommended_items.sort_values(ascending=False).head(top_n)

In [31]:
recommendations = recommend_items(user=17)
print(recommendations)

movieId
3      5.0
351    5.0
dtype: float64


In [32]:
train_data, test_data = train_test_split(XGT_rec.T, test_size=.25, random_state=42)

In [33]:
model_rec = AlternatingLeastSquares(factors=64)
model_rec.fit(train_data)

  0%|          | 0/15 [00:00<?, ?it/s]

In [34]:
precision_at_k(model_rec, train_data, test_data)

  0%|          | 0/2411 [00:00<?, ?it/s]

0.08006912442396313

In [35]:
def novelty(train_rec, preds):
    item_sum = (train_rec > 0.5).sum(axis=0)
    item_proba = item_sum / item_sum.sum()
    result = []
    for x in preds[0]:
        for y in x:
            result.append(np.log2(1+item_proba[0, y]))
    
    return np.mean(result)

In [36]:
user_id = list(range(test_data.shape[0]))

In [37]:
predicts = model_rec.recommend(user_id, test_data)

In [38]:
predicts[0]

array([[364, 110,  50, ..., 533, 563, 118],
       [473,  27, 198, ..., 274, 552, 218],
       [110, 605, 413, ..., 287, 306, 437],
       ...,
       [413, 386, 473, ..., 248, 599,  27],
       [  9,   8,   7, ...,   2,   1,   0],
       [609, 176, 231, ...,  61, 476, 595]], dtype=int32)

In [39]:
novelty(train_data, predicts)

np.float64(0.008192414702018017)